In [55]:
from csv import reader
import pdfplumber
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import glob
import os
from collections import Counter
import pdfplumber
import string

pdf_folder = "Document_Corpus"
corpus = []
metadata = []
remove_table = str.maketrans("", "", string.punctuation)

pdf_files = glob.glob(os.path.join(pdf_folder, "*.pdf"))

for pdf_path in pdf_files:
    filename = os.path.basename(pdf_path)

    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages):
                text = page.extract_text(x_tolerance=2.0)
                if text and text.strip():
                    corpus.append(text)
                    metadata.append(
                        {
                            "filename": filename,
                            "page": page_num + 1,
                            "raw_text": text.strip(),
                        }
                    )

    except Exception as e:
        print(f"Error reading{filename}:{e}")
print(f"Loaded {len(corpus)} pages from {len(pdf_files)} PDF Files")

Loaded 92 pages from 4 PDF Files


In [56]:
vectorizer = TfidfVectorizer()
Tf_Idfmatrix = vectorizer.fit_transform(corpus)

In [57]:
def search(query, top_n=5):
    query_vector = vectorizer.transform([query])
    similiarity_scores = cosine_similarity(query_vector, Tf_Idfmatrix).flatten()
    top_idx = np.argsort(similiarity_scores)[::-1][:top_n]

    match_found = False
    for rank, idx in enumerate(top_idx):
        score = similiarity_scores[idx]
        if score > 0:
            match_found = True

            if "metadata" in globals() and len(metadata) > idx:
                meta = metadata[idx]
                print(f"Rank {rank + 1} - Score {score:.2f}")
                print(f"File: {meta['filename']}(Page {meta['page']})")
                snippet = meta["raw_text"].replace("\n", "")[:200]
                print(f"Snippet: {snippet}...")

            else:
                print(f"rank {rank+1} - Score {score:.2f}")
                print(f"{corpus[idx]}")

    if not match_found:
        print("No matching document found with score > 0.")
    
    return top_idx

In [58]:
results = search("Autoencoder")

Rank 1 - Score 0.13
File: DEEP SEMI-SUPERVISED ANOMALY DETECTION.pdf(Page 19)
Snippet: Published as a conference paper at ICLR 2020Note that we use the latent class probability estimate (normal vs. anomalous) of semi-supervisedDGM as a natural choice for the anomaly score, and not the r...
Rank 2 - Score 0.08
File: GANomaly Semi-Supervised Anomaly.pdf(Page 3)
Snippet: GANomaly 3Schlegl et al. [39] hypothesize that the latent vector of a GAN representsthe true distribution of the data and remap to the latent vector by optimizinga pre-trained GAN based on the latent ...
Rank 3 - Score 0.05
File: GANomaly Semi-Supervised Anomaly.pdf(Page 4)
Snippet: 4 S. Akcay et al.the model predicts the future frame of possible standard example, which dis-tinguishes the abnormality during the inference. In another study on the sametask, Hasan et al. [18] consid...
Rank 4 - Score 0.05
File: DEEP SEMI-SUPERVISED ANOMALY DETECTION.pdf(Page 17)
Snippet: Published as a conference paper at ICLR 2020C OPTIMIZA